In [20]:
import pandas as pd
import joblib
import os

# paths
MODEL_PATH = '../models'
DATA_PATH = './data'  # test.csv is inside the notebooks/data folder

In [21]:
#load model and test data

lr_model = joblib.load(os.path.join(MODEL_PATH, 'linear_regression_model.pkl'))

test_data = pd.read_csv(os.path.join(DATA_PATH, 'test.csv'))

In [22]:
# PREPROCESSING 
# Fill missing numerical values (same as training)
num_cols = ['LotFrontage', 'MasVnrArea', 'GarageYrBlt']  # add any other numeric columns with NaNs
for col in num_cols:
    if col in test_data.columns:
        test_data[col] = test_data[col].fillna(0)

# Fill categorical columns with 'None'
cat_cols = ['Alley', 'FireplaceQu', 'PoolQC', 'Fence', 'MiscFeature', 
            'MasVnrType', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 
            'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']
for col in cat_cols:
    if col in test_data.columns:
        test_data[col] = test_data[col].fillna('None')

# Fill remaining categorical columns with mode
for col in test_data.select_dtypes(include='object').columns:
    test_data[col] = test_data[col].fillna(test_data[col].mode()[0])

C:\Users\HP\AppData\Local\Temp\ipykernel_1732\2472960106.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in test_data.select_dtypes(include='object').columns:


In [28]:
# ONE-HOT ENCODING 
X_test_encoded = pd.get_dummies(test_data)


In [29]:
#  align columns with training data 
# Load training columns used in model
train_columns = joblib.load(os.path.join(MODEL_PATH, 'training_columns.pkl'))

# Add missing columns in test data
for col in train_columns:
    if col not in X_test_encoded.columns:
        X_test_encoded[col] = 0

# Ensure the order of columns is the same as training
X_test_encoded = X_test_encoded[train_columns]

C:\Users\HP\AppData\Local\Temp\ipykernel_1732\2004365970.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_test_encoded[col] = 0


In [31]:
# ENSURE NO NaNs 
# Fill any remaining numeric NaNs with 0
X_test_encoded = X_test_encoded.fillna(0)

# Confirm no NaNs remain
if X_test_encoded.isnull().sum().sum() > 0:
    print("Warning: There are still missing values!")
else:
    print("No missing values in test data. Ready for predictions.")

No missing values in test data. Ready for predictions.


In [32]:
# make predictions
predictions = lr_model.predict(X_test_encoded)

In [33]:
#  SHOW EXAMPLE PREDICTIONS 
print("First 10 predictions:", predictions[:10])

First 10 predictions: [115344.60358057 163812.62791128 180944.12196123 198498.31308533
 209040.6943161  163546.24602877 167883.20286926 147838.45908401
 197884.1128656  111036.99193887]
